# Pandas: Handling Missing Data — `replace()` and `interpolate()`

## Learning objectives
By the end of these notes, you should be able to:
- Identify missing values in a DataFrame.
- Replace specific values and placeholder strings using `replace()`.
- Understand the difference between `replace()`, `fillna()`, `ffill()`, `bfill()`, and `interpolate()`.
- Use interpolation parameters such as `method`, `axis`, `limit`, `limit_direction`, and `limit_area`.
- Decide when a missing value should or should not be estimated.

**Important:** Always keep the original data when experimenting. A method that works for one column may be inappropriate for another.

## 1. Setup

Pandas is a Python library for working with tabular data. The alias `pd` is a common convention.

If pandas is not installed in your Python environment, run this in a notebook cell:

```python
%pip install pandas
```

In [2]:
import pandas as pd

## 2. Recognizing missing data

Pandas commonly represents missing values with `NaN`, `None`, or `pd.NA`, depending on the data type.

Useful checks:
- `df.isna()` returns `True` where values are missing.
- `df.isna().sum()` counts missing values in each column.
- `df.notna()` returns `True` for values that are present.

A blank string (`""`) or a string such as `"N/A"` is not automatically treated as a missing value in every situation. You may need to convert such placeholders explicitly.

In [3]:
df = pd.DataFrame({
    "name": ["Asha", "Rohan", "Meera", "Kabir"],
    "score": [78, None, 91, None],
    "city": ["Pune", "N/A", "Mumbai", "Pune"]
})

display(df)
print("Missing values per column:")
display(df.isna().sum())

,name,score,city
0,Asha,78.0,Pune
1,Rohan,NaN,N/A
2,Meera,91.0,Mumbai
3,Kabir,NaN,Pune


Missing values per column:


name     0
score    2
city     0
dtype: int64

## 3. `replace()` — replace values you specify

`replace()` changes values that match a value or pattern you specify. It is useful for correcting inconsistent labels, replacing placeholder text, or mapping known codes to new values.

### 3.1 Replace one value

Syntax: `df.replace(to_replace=old_value, value=new_value)`

This returns a changed object by default; it does not change the original DataFrame unless you assign the result back or use an in-place option.

In [4]:
cities = pd.DataFrame({"city": ["Nagpur", "Mumbai", "Nagpur", "Pune"]})

updated_cities = cities.replace("Nagpur", "Ratnagiri")
display(cities)          # Original remains unchanged
display(updated_cities)  # Replaced version

,city
0,Nagpur
1,Mumbai
2,Nagpur
3,Pune


,city
0,Ratnagiri
1,Mumbai
2,Ratnagiri
3,Pune


### 3.2 Replace multiple values

Pass lists of old values and replacement values in matching order. Make sure the two lists correspond as intended.

In [5]:
codes = pd.DataFrame({"code": [1001, 1002, 1001, 1003]})

updated_codes = codes.replace([1001, 1002], [1, 2])
display(updated_codes)

,code
0,1
1,2
2,1
3,1003


### 3.3 Replace values in specific columns using a dictionary

A nested dictionary lets you define different replacements for different columns. This is helpful when the same word or code should be changed only in one column.

In [6]:
df = pd.DataFrame({
    "city": ["Nagpur", "Mumbai", "Nagpur"],
    "channel": ["Online", "Store", "Online"]
})

cleaned = df.replace({
    "city": {"Nagpur": "Ratnagiri"},
    "channel": {"Online": "Digital"}
})
display(cleaned)

,city,channel
0,Ratnagiri,Digital
1,Mumbai,Store
2,Ratnagiri,Digital


### 3.4 Regex replacement

Regular expressions (regex) describe text patterns. With `regex=True`, `replace()` can replace values matching a pattern.

In this example, a single uppercase letter in the `category` column is replaced with `"Unknown"`. Be careful: broad patterns can replace more values than intended.

In [7]:
categories = pd.DataFrame({"category": ["A", "B", "C", None, "Retail"]})

result = categories.replace(
    {"category": r"^[A-Z]$"},
    "Unknown",
    regex=True
)
display(result)

,category
0,Unknown
1,Unknown
2,Unknown
3,NaN
4,Retail


### 3.5 Convert placeholder text into genuine missing values

A dataset may use strings such as `"N/A"`, `"missing"`, or `""` to indicate absent data. Convert them to `pd.NA` before using missing-value tools.

`replace()` is suitable here because you are replacing known placeholder strings.

In [8]:
ratings = pd.DataFrame({"rating": ["5", "N/A", "missing", "3", ""]})

ratings_clean = ratings.replace(["N/A", "missing", ""], pd.NA)
display(ratings_clean)
display(ratings_clean.isna().sum())

,rating
0,5
1,NaN
2,NaN
3,3
4,NaN


rating    3
dtype: int64

## 4. `interpolate()` — estimate missing numeric values

Interpolation estimates missing values from known values around them. It is often useful for measurements recorded over time when values are expected to change gradually.

For example, if a temperature reading is 22 on day 1 and 26 on day 3, linear interpolation estimates 24 for day 2.

**Do not assume interpolation is always correct.** A missing transaction amount or a student's real exam score is not necessarily something that should be estimated.

In [9]:
temperatures = pd.DataFrame({
    "day": [1, 2, 3, 4, 5],
    "temperature": [20.0, 22.0, None, 26.0, 28.0]
})

display(temperatures)
display(temperatures.interpolate(method="linear"))

,day,temperature
0,1,20.0
1,2,22.0
2,3,NaN
3,4,26.0
4,5,28.0


,day,temperature
0,1,20.0
1,2,22.0
2,3,24.0
3,4,26.0
4,5,28.0


### 4.1 `method="linear"`

Linear interpolation treats rows as equally spaced and estimates values along a straight line between known observations.

If missing values occur between 22 and 28 with two equal steps, the values are estimated as 24 and 26.

Note: `method="linear"` uses the row positions rather than the actual spacing of index values. For time-indexed data, other methods may be appropriate depending on the index and use case.

In [10]:
example = pd.DataFrame({
    "reading": [22.0, None, None, 28.0]
})

display(example.interpolate(method="linear"))

,reading
0,22.0
1,24.0
2,26.0
3,28.0


### 4.2 `axis`

- `axis=0` (default): interpolate down rows, column by column.
- `axis=1`: interpolate across columns, row by row.

Choose the axis based on how the data is organized. Across-column interpolation only makes sense if the column order has a meaningful numeric progression.

In [11]:
measurements = pd.DataFrame({
    "morning": [10.0, None, 30.0],
    "afternoon": [20.0, 40.0, 60.0],
    "evening": [30.0, 50.0, 70.0]
})

print("Down rows (axis=0):")
display(measurements.interpolate(axis=0))

print("Across columns (axis=1):")
display(measurements.interpolate(axis=1))

Down rows (axis=0):


,morning,afternoon,evening
0,10.0,20.0,30.0
1,20.0,40.0,50.0
2,30.0,60.0,70.0


Across columns (axis=1):


,morning,afternoon,evening
0,10.0,20.0,30.0
1,NaN,40.0,50.0
2,30.0,60.0,70.0


### 4.3 `limit`

`limit` sets the maximum number of consecutive missing values to fill in a run. For example, `limit=1` limits filling to one missing cell in a consecutive gap.

In [12]:
readings = pd.DataFrame({"value": [10.0, None, None, None, 40.0]})

print("No limit:")
display(readings.interpolate())

print("At most one consecutive missing value:")
display(readings.interpolate(limit=1))

No limit:


,value
0,10.0
1,17.5
2,25.0
3,32.5
4,40.0


At most one consecutive missing value:


,value
0,10.0
1,17.5
2,NaN
3,NaN
4,40.0


### 4.4 `limit_direction`

This controls the direction in which the `limit` is applied:
- `"forward"`
- `"backward"`
- `"both"`

Try the options and compare which missing cells remain.

In [13]:
readings = pd.DataFrame({"value": [10.0, None, None, None, 40.0]})

display(readings.interpolate(limit=1, limit_direction="forward"))
display(readings.interpolate(limit=1, limit_direction="backward"))
display(readings.interpolate(limit=1, limit_direction="both"))

,value
0,10.0
1,17.5
2,NaN
3,NaN
4,40.0


,value
0,10.0
1,NaN
2,NaN
3,32.5
4,40.0


,value
0,10.0
1,17.5
2,NaN
3,32.5
4,40.0


### 4.5 `limit_area`

- `limit_area="inside"`: fill missing values only when they are surrounded by valid values.
- `limit_area="outside"`: fill missing values outside the range of valid observations, where supported by the selected interpolation behavior.

With linear interpolation, edge values are generally not extrapolated by default, so some leading or trailing missing values may remain missing.

In [14]:
edge_readings = pd.DataFrame({"reading": [None, 10.0, None, 30.0, None]})

print("Inside only:")
display(edge_readings.interpolate(limit_area="inside"))

print("Outside only:")
display(edge_readings.interpolate(limit_area="outside"))

Inside only:


,reading
0,NaN
1,10.0
2,20.0
3,30.0
4,NaN


Outside only:


,reading
0,NaN
1,10.0
2,NaN
3,30.0
4,30.0


## 5. `ffill()` and `bfill()`

These methods copy existing values rather than estimating a number:
- `ffill()` (forward fill) copies the previous valid value into following missing cells.
- `bfill()` (backward fill) copies the next valid value into earlier missing cells.

They can be useful for values that remain in effect until updated, but they are not automatically appropriate for every dataset.

In [15]:
channels = pd.DataFrame({"channel": ["Online", None, None, "Store", None]})

print("Forward fill:")
display(channels.ffill())

print("Backward fill:")
display(channels.bfill())

print("Forward fill with limit=1:")
display(channels.ffill(limit=1))

Forward fill:


,channel
0,Online
1,Online
2,Online
3,Store
4,Store


Backward fill:


,channel
0,Online
1,Store
2,Store
3,Store
4,NaN


Forward fill with limit=1:


,channel
0,Online
1,Online
2,NaN
3,Store
4,Store


## 6. `replace()` vs `fillna()` vs filling vs interpolation

| Method | Main purpose | Example use |
|---|---|---|
| `replace()` | Replace specified values or patterns | Change `"N/A"` to `pd.NA`; correct `"Naggpur"` |
| `fillna()` | Fill missing cells with a chosen value or statistic | Fill missing category with `"Not provided"` |
| `ffill()` | Carry previous valid value forward | A status that remains in effect until changed |
| `bfill()` | Carry next valid value backward | Fill from the next known label when justified |
| `interpolate()` | Estimate missing numeric values from neighboring values | Missing temperature readings in a gradual time series |

The key question is: **what does the missing value mean, and is it reasonable to replace or estimate it?**

## 7. Practice dataset 1 — Sales records

This dataset includes missing sales amounts, units, city names, and category labels. It is useful for practicing missing-value inspection and replacement.

Run this cell to create the DataFrame.

In [16]:
sales = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "city": ["Nagpur", "Mumbai", "Nagpur", None, "Pune", "Mumbai", "Nashik", "Pune"],
    "channel": ["Online", "Store", "Online", "Store", None, "Online", "Store", "Online"],
    "category": ["A", "B", "A", "C", "B", None, "A", "C"],
    "sales_amount": [1200.0, None, 1800.0, 1500.0, None, 2100.0, 1900.0, None],
    "units_sold": [2, 1, None, 3, 2, None, 4, 2]
})

display(sales)
display(sales.isna().sum())

,order_id,city,channel,category,sales_amount,units_sold
0,1001,Nagpur,Online,A,1200.0,2.0
1,1002,Mumbai,Store,B,NaN,1.0
2,1003,Nagpur,Online,A,1800.0,NaN
3,1004,NaN,Store,C,1500.0,3.0
4,1005,Pune,NaN,B,NaN,2.0
5,1006,Mumbai,Online,NaN,2100.0,NaN
6,1007,Nashik,Store,A,1900.0,4.0
7,1008,Pune,Online,C,NaN,2.0


order_id        0
city            1
channel         1
category        1
sales_amount    3
units_sold      2
dtype: int64

### Practice tasks — Sales records

Try these yourself before checking any solution:
1. Replace `"Nagpur"` with `"Ratnagiri"` in the `city` column only.
2. Fill missing cities with `"Not Provided"`.
3. Fill missing categories with `"Unclassified"`.
4. Compare `sales_amount` before and after linear interpolation.
5. Explain why interpolation may not be suitable for missing sales transactions.
6. Make a copy before cleaning so you can compare the original and cleaned data.

In [17]:
# Write your answers below.
# Hint: use sales.copy() so the original data stays unchanged.

sales_clean = sales.copy()

# Task 1:
# sales_clean["city"] = ...

# Task 2:
# sales_clean["city"] = ...

# Task 3:
# ...

# Task 4:
# ...

## 8. Practice dataset 2 — Sensor readings

Sensor readings are recorded daily. Since temperature and humidity often change gradually, interpolation can be a reasonable experiment—but real sensor behavior and gaps should still be checked.

In [18]:
sensor = pd.DataFrame({
    "timestamp": pd.date_range("2026-01-01", periods=10, freq="D"),
    "temperature_c": [22.0, 22.5, None, 23.5, None, 24.0, 24.5, None, 25.0, 25.5],
    "humidity_pct": [60.0, None, 62.0, 63.0, 64.0, None, 66.0, 67.0, None, 69.0]
})

display(sensor)
display(sensor.isna().sum())

,timestamp,temperature_c,humidity_pct
0,2026-01-01,22.0,60.0
1,2026-01-02,22.5,NaN
2,2026-01-03,NaN,62.0
3,2026-01-04,23.5,63.0
4,2026-01-05,NaN,64.0
5,2026-01-06,24.0,NaN
6,2026-01-07,24.5,66.0
7,2026-01-08,NaN,67.0
8,2026-01-09,25.0,NaN
9,2026-01-10,25.5,69.0


timestamp        0
temperature_c    3
humidity_pct     3
dtype: int64

### Practice tasks — Sensor readings

1. Interpolate `temperature_c` and store the result in a new column named `temperature_filled`.
2. Interpolate `humidity_pct` and compare the results.
3. Try `limit=1`.
4. Try `limit_area="inside"`.
5. Explain what happens to a missing value at the beginning or end of a column.
6. Keep the original columns so you can see which values were estimated.

In [19]:
sensor_practice = sensor.copy()

# Write your code below:
# sensor_practice["temperature_filled"] = ...
# sensor_practice["humidity_filled"] = ...

display(sensor_practice)

,timestamp,temperature_c,humidity_pct
0,2026-01-01,22.0,60.0
1,2026-01-02,22.5,NaN
2,2026-01-03,NaN,62.0
3,2026-01-04,23.5,63.0
4,2026-01-05,NaN,64.0
5,2026-01-06,24.0,NaN
6,2026-01-07,24.5,66.0
7,2026-01-08,NaN,67.0
8,2026-01-09,25.0,NaN
9,2026-01-10,25.5,69.0


## 9. Practice dataset 3 — Student performance

This dataset contains missing marks and attendance values. Use it to practice, but think carefully before estimating real student scores: an estimated value is not the same as an observed score.

In [20]:
students = pd.DataFrame({
    "student": ["Asha", "Rohan", "Meera", "Kabir", "Isha", "Dev", "Sara", "Arjun"],
    "maths": [78.0, None, 91.0, 65.0, None, 88.0, 72.0, 95.0],
    "science": [82.0, 76.0, None, 69.0, 85.0, None, 79.0, 93.0],
    "attendance_pct": [95.0, 88.0, 92.0, None, 97.0, 84.0, None, 99.0]
})

display(students)
display(students.isna().sum())

,student,maths,science,attendance_pct
0,Asha,78.0,82.0,95.0
1,Rohan,NaN,76.0,88.0
2,Meera,91.0,NaN,92.0
3,Kabir,65.0,69.0,NaN
4,Isha,NaN,85.0,97.0
5,Dev,88.0,NaN,84.0
6,Sara,72.0,79.0,NaN
7,Arjun,95.0,93.0,99.0


student           0
maths             2
science           2
attendance_pct    2
dtype: int64

### Practice tasks — Student performance

1. Count missing values by column.
2. Create a copy and try interpolation on `maths`.
3. Compare it with filling missing marks using the column mean.
4. Explain why both methods could misrepresent a student's actual performance.
5. Choose a method for missing attendance values and justify your choice.

In [21]:
students_practice = students.copy()

# Write your code below:
# students_practice["maths_interpolated"] = ...
# students_practice["maths_mean_filled"] = ...

display(students_practice)

,student,maths,science,attendance_pct
0,Asha,78.0,82.0,95.0
1,Rohan,NaN,76.0,88.0
2,Meera,91.0,NaN,92.0
3,Kabir,65.0,69.0,NaN
4,Isha,NaN,85.0,97.0
5,Dev,88.0,NaN,84.0
6,Sara,72.0,79.0,NaN
7,Arjun,95.0,93.0,99.0


## 10. Quick revision

Before finishing, make sure you can explain these points in your own words:

1. What does `replace()` do?
2. How is `replace()` different from `fillna()`?
3. What does linear interpolation estimate?
4. What is the difference between `ffill()` and `bfill()`?
5. What does `limit=1` mean in an interpolation example?
6. What do `limit_direction` and `limit_area` control?
7. Why should you keep a copy of the original dataset?
8. Give one example where interpolation may be useful and one where it may be misleading.

### Mini challenge

Load a CSV, inspect missing values, choose an appropriate method for each column, create a cleaned copy, and explain your decisions in Markdown cells. This is a useful format for your pandas learning notes on GitHub.